In [1]:
import yfinance as yf
import pandas as pd
import fetch_stock_data as data
import stock_model_trainer as trainer
import stock_predictor_models as models
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from itertools import cycle

# Collect and Preprocess Data

In [2]:
# list of symbols
sp500_symbols = data.get_sp500_symbols()
# collect entire window (train, test, eval)
df_all = data.collect_data(sp500_symbols, "2015-01-01", "2025-10-31", freq="1d")
# drop any stocks missing any days, leaves 401 stocks
df_all = df_all.dropna(axis=1, how="any")
# convert to pct change
df_all = df_all.pct_change().dropna()
# normalize
mean_train = df_all.loc[df_all.index < "2024-01-01"].mean(axis=0)
std_train = df_all.loc[df_all.index < "2024-01-01"].std(axis=0)
df_all = (df_all - mean_train) / std_train
# train is 2015-01-01 to 2024-01-01
pandas_df_train = df_all.loc[df_all.index < "2024-01-01"]
# test is 2024-01-01 to 2024-10-31
pandas_df_test = df_all.loc[(df_all.index >= "2024-01-01") & (df_all.index <= "2024-10-31")]
# holdout is 2024-11-01 to 2025-10-31
pandas_df_holdout = df_all.loc[(df_all.index >= "2024-11-01") & (df_all.index <= "2025-10-31")]
# convert to torch tensors
prices_train = torch.tensor(pandas_df_train.values, dtype=torch.float32)
prices_test = torch.tensor(pandas_df_test.values, dtype=torch.float32)
prices_holdout = torch.tensor(pandas_df_holdout.values, dtype=torch.float32)

c:\Users\rkved\CS7643\CS7643-Final-Project\fetch_stock_data.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbols, start=start_date, end=end_date, interval=freq)
[*********************100%***********************]  505 of 505 completed

74 Failed downloads:
['JEC', 'TIF', 'JWN', 'CELG', 'COG', 'HES', 'ABC', 'DFS', 'RTN', 'TMK', 'ANSS', 'BLL', 'ARNC', 'WRK', 'FBHS', 'LLL', 'DWDP', 'XLNX', 'RE', 'TSS', 'MON', 'FLIR', 'CXO', 'PBCT', 'RHT', 'KSU', 'MYL', 'WBA', 'DISCA', 'XL', 'GPS', 'PXD', 'BHGE', 'BRK.B', 'ETFC', 'MRO', 'NLSN', 'FL', 'DISH', 'APC', 'SYMC', 'CTXS', 'PDCO', 'CERN', 'DISCK', 'HCP', 'ANTM', 'NBL', 'CBS', 'DRE', 'AGN', 'VIAB', 'ADS', 'VAR', 'HRS', 'UTX', 'PKI', 'CTL', 'ATVI', 'CHK', 'WLTW', 'XEC', 'ALXN', 'JNPR']: YFTzMissingError('possibly delisted; no timezone found')
['DPS', 'CBG', 'SNI', 'WYN', 'KORS', 'BF.B', 'HCN', 'LUK', 'GGP', 'SRCL']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-01-01

In [3]:
# need a map from the evaluation stocks to their column position in the data
eval_stocks = data.get_eval_stocks()
num_stocks_out = len(eval_stocks)
stock_index_map = {c: df_all.columns.get_loc(c) for c in eval_stocks}
stock_indices = [stock_index_map[stock] for stock in eval_stocks]

In [4]:
# hold onto mean and std of train for only the eval stocks
# this helps to undo transformation for just the eval stocks
eval_mean_train = mean_train.iloc[stock_indices].values
eval_std_train = std_train.iloc[stock_indices].values

In [5]:
# turn raw data into sequences
# X will be shape (batches, seq_len, num_stocks)
# y will be shape (batches, num_stocks)
seq_len = 60
X_train, y_train = trainer.create_sequences(prices_train, seq_len)
X_test, y_test = trainer.create_sequences(prices_test, seq_len)
X_holdout, y_holdout = trainer.create_sequences(prices_holdout, seq_len)
_, seq_len, num_stocks_in = X_train.shape

In [6]:
# select the eval columns
y_train = y_train[:, stock_indices]
y_test = y_test[:, stock_indices]
y_holdout = y_holdout[:, stock_indices]

# Train

In [7]:
# setup for training
criterion = nn.HuberLoss()
model = models.StockCNN(
    input_dim=num_stocks_in,
    output_dim=num_stocks_out, # predict only the 5 we care about
    hidden=32
)
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=20,
    gamma=0.1
)
model_trainer = trainer.ModelTrainer(
    criterion,
    model,
    optimizer,
    X_train,
    y_train,
    X_test,
    y_test,
    # scheduler=scheduler,
)

In [8]:
model_trainer.train(num_epochs=50)
model_trainer.plot_losses(fig_path="sp500_cnn/losses.png")

Epoch [5/50], Train Loss: 0.346654, Test Loss: 0.281946
Epoch [10/50], Train Loss: 0.345339, Test Loss: 0.283308
Epoch [15/50], Train Loss: 0.344261, Test Loss: 0.282215
Epoch [20/50], Train Loss: 0.343151, Test Loss: 0.282957
Epoch [25/50], Train Loss: 0.341989, Test Loss: 0.282270
Epoch [30/50], Train Loss: 0.340669, Test Loss: 0.282495
Epoch [35/50], Train Loss: 0.339194, Test Loss: 0.282759
Epoch [40/50], Train Loss: 0.337886, Test Loss: 0.283424
Epoch [45/50], Train Loss: 0.336998, Test Loss: 0.282034
Epoch [50/50], Train Loss: 0.335687, Test Loss: 0.282576


<Figure size 640x480 with 0 Axes>

# Eval on Test Set

In [9]:
model.eval()
with torch.no_grad():
    pred_test_scaled = model(X_test)
pred_test_unscaled = pred_test_scaled * eval_std_train + eval_mean_train
y_test_unscaled = y_test * eval_std_train + eval_mean_train

In [10]:
# Get the default matplotlib color cycle
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])
for i, stock in enumerate(eval_stocks):
    color = next(color_cycle)
    plt.plot(pred_test_unscaled[:, i], label=f"{stock}_Pred", color=color)
    plt.plot(y_test_unscaled[:, i], label=f"{stock}_Actual", color=color, linestyle="--")
    plt.title(f"{stock} Daily Returns Predicted vs Actual")
    plt.legend()
    plt.savefig(f"outputs/sp500_cnn/{stock}_test_preds.png")
    plt.clf()

<Figure size 640x480 with 0 Axes>

In [11]:
# MAE by stock
torch.mean(torch.abs(pred_test_unscaled - y_test_unscaled), dim=0)

tensor([0.0109, 0.0101, 0.0100, 0.0159, 0.0109], dtype=torch.float64)

In [12]:
eval_stocks

['AAPL', 'JPM', 'XOM', 'BA', 'UNH']

In [13]:
# MAE total
torch.mean(torch.abs(pred_test_unscaled - y_test_unscaled))

tensor(0.0116, dtype=torch.float64)

In [14]:
# directional accuracy by stock
(pred_test_unscaled * y_test_unscaled > 0).to(torch.float).mean(dim=0)

tensor([0.6225, 0.4967, 0.4901, 0.5364, 0.4834])

In [15]:
# directional accuracy overall
(pred_test_unscaled * y_test_unscaled > 0).to(torch.float).mean()

tensor(0.5258)

# Eval on Holdout set

In [16]:
model.eval()
with torch.no_grad():
    pred_holdout_scaled = model(X_holdout)
pred_holdout_unscaled = pred_holdout_scaled * eval_std_train + eval_mean_train
y_holdout_unscaled = y_holdout * eval_std_train + eval_mean_train

In [17]:
# Get the default matplotlib color cycle
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])
for i, stock in enumerate(eval_stocks):
    color = next(color_cycle)
    plt.plot(pred_holdout_unscaled[:, i], label=f"{stock}_Pred", color=color)
    plt.plot(y_holdout_unscaled[:, i], label=f"{stock}_Actual", color=color, linestyle="--")
    plt.title(f"{stock} Daily Returns Predicted vs Actual")
    plt.legend()
    plt.savefig(f"outputs/sp500_cnn/{stock}_holdout_preds.png")
    plt.clf()

<Figure size 640x480 with 0 Axes>

In [18]:
# MAE by stock
torch.mean(torch.abs(pred_holdout_unscaled - y_holdout_unscaled), dim=0)

tensor([0.0142, 0.0110, 0.0115, 0.0158, 0.0190], dtype=torch.float64)

In [19]:
# MAE total
torch.mean(torch.abs(pred_test_unscaled - y_test_unscaled))

tensor(0.0116, dtype=torch.float64)

In [20]:
# directional accuracy by stock
(pred_holdout_unscaled * y_holdout_unscaled > 0).to(torch.float).mean(dim=0)

tensor([0.4709, 0.5979, 0.5397, 0.5661, 0.5291])

In [21]:
# directional accuracy overall
(pred_holdout_unscaled * y_holdout_unscaled > 0).to(torch.float).mean()

tensor(0.5407)